# 07_frecuencias_ofensivas_y_ngrams

## Objetivo

Identificar palabras y expresiones asociadas a hostilidad en los comentarios del corpus formal mediante cuatro recursos complementarios:

1. términos categorizados del lexicón;
2. validación contra `manual_hostility` y `manual_hate_speech`;
3. unigramas, bigramas y trigramas contrastados entre comentarios hostiles y no hostiles;
4. frecuencias exploratorias dentro del corpus predicho como hostil.

El notebook no considera ofensiva una palabra únicamente porque aparezca en un lexicón. Los resultados se presentan como **candidatos para interpretación contextual**.


## 1. Entradas y salidas

### Entradas
- `reports/formal_eda/manual_review_sample.csv`
- `lexicons/processed/hatecr_lexicon.csv`
- `data/processed/x_media_anchored_interactions_corpus_formal_with_hostility_and_experimental_hate_predictions.csv`

### Salidas
- tablas en `reports/formal_lexical/`
- gráficos en `reports/formal_lexical/figures/`
- lista de candidatos para revisión manual, sin modificar las etiquetas existentes


In [ ]:
import importlib
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def is_project_dir(path):
    return (path / "config").exists() and (path / "data").exists() and (path / "notebooks").exists()


def find_project_root(start):
    for candidate in [start] + list(start.parents):
        if is_project_dir(candidate):
            return candidate
        child = candidate / "HateCR"
        if is_project_dir(child):
            return child
    return start


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.labels as label_utils
import src.text_analysis as text_analysis
importlib.reload(label_utils)
importlib.reload(text_analysis)

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
FORMAL_EDA_REPORTS = PROJECT_ROOT / "reports" / "formal_eda"
REPORTS_DIR = PROJECT_ROOT / "reports" / "formal_lexical"
FIGURES_DIR = REPORTS_DIR / "figures"
LEXICON_DIR = PROJECT_ROOT / "lexicons" / "processed"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

MANUAL_SAMPLE_PATH = FORMAL_EDA_REPORTS / "manual_review_sample.csv"
LEXICON_PATH = LEXICON_DIR / "hatecr_lexicon.csv"
CORPUS_PATH = DATA_PROCESSED / "x_media_anchored_interactions_corpus_formal_with_hostility_and_experimental_hate_predictions.csv"

MIN_MANUAL_TERM_DOCUMENTS = int(os.getenv("MIN_MANUAL_TERM_DOCUMENTS", "3"))
MIN_HOSTILITY_PRECISION = float(os.getenv("MIN_HOSTILITY_PRECISION", "0.60"))
MIN_HOSTILITY_LIFT = float(os.getenv("MIN_HOSTILITY_LIFT", "1.25"))
MIN_HIGH_CONFIDENCE_PRECISION = float(os.getenv("MIN_HIGH_CONFIDENCE_PRECISION", "1.0"))
TOP_N = int(os.getenv("LEXICAL_TOP_N", "30"))

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.grid"] = True

print("PROJECT_ROOT:", PROJECT_ROOT)
print("REPORTS_DIR:", REPORTS_DIR)
print("API y descargas: desactivadas")


In [ ]:
def safe_read_csv(path, name, **kwargs):
    if not path.exists():
        raise FileNotFoundError(f"Falta {name}: {path}")
    for encoding in ["utf-8-sig", "utf-8", "latin-1"]:
        try:
            frame = pd.read_csv(path, encoding=encoding, **kwargs)
            for id_column in ["tweet_id", "review_id", "source_post_id"]:
                if id_column in frame.columns:
                    frame[id_column] = frame[id_column].astype("string")
            print(f"[OK] {name}: {len(frame):,} filas")
            return frame
        except UnicodeDecodeError:
            continue
    raise ValueError(f"No se pudo leer {path}")


manual_raw_df = safe_read_csv(MANUAL_SAMPLE_PATH, "muestra manual")
lexicon_df = safe_read_csv(LEXICON_PATH, "lexicón procesado", dtype=str)
corpus_df = safe_read_csv(CORPUS_PATH, "corpus con predicciones")

manual_df, manual_diagnostics = label_utils.prepare_manual_annotations(
    manual_raw_df,
    strict=True,
)
manual_df = manual_df[manual_df["annotation_ready_for_training"]].copy()

required_corpus_columns = {
    "tweet_id", "text", "event_id", "anchor_media_handle",
    "lexicon_terms_found", "ml_hostility_pred",
}
missing = required_corpus_columns.difference(corpus_df.columns)
if missing:
    raise KeyError(f"Faltan columnas del corpus: {sorted(missing)}")

print("Muestra manual lista:", len(manual_df))
print("Corpus único:", corpus_df["tweet_id"].nunique())


## 2. Índice categorizado y validación manual de términos

Se excluyen entradas `uncategorized`. Cada término se cuenta una sola vez por comentario. La tabla resultante informa frecuencia, precisión manual, asociación relativa (`lift`), categorías y fuentes.


In [ ]:
term_index_df = text_analysis.build_categorized_lexicon_index(lexicon_df)

manual_match_columns = [
    "tweet_id", "event_id", "anchor_media_handle",
    "manual_hostility_normalized", "manual_hate_speech_normalized",
]
manual_matches_df = text_analysis.explode_categorized_matches(
    manual_df,
    term_index_df,
    id_columns=manual_match_columns,
)

full_match_columns = [
    "tweet_id", "event_id", "anchor_media_handle",
    "ml_hostility_pred", "ml_hate_speech_pred_experimental",
]
full_matches_df = text_analysis.explode_categorized_matches(
    corpus_df,
    term_index_df,
    id_columns=full_match_columns,
)

base_hostility_rate = float(manual_df["manual_hostility_normalized"].mean())
term_validation_df = text_analysis.build_manual_term_validation(
    manual_matches_df,
    document_column="tweet_id",
    hostility_column="manual_hostility_normalized",
    hate_column="manual_hate_speech_normalized",
    total_hostility_rate=base_hostility_rate,
)

full_term_frequency_df = (
    full_matches_df.groupby("term_norm", as_index=False)
    .agg(
        full_document_count=("tweet_id", "nunique"),
        predicted_hostile_document_count=("ml_hostility_pred", "sum"),
        predicted_hate_document_count=("ml_hate_speech_pred_experimental", "sum"),
    )
)
term_validation_df = term_validation_df.merge(
    full_term_frequency_df,
    on="term_norm",
    how="left",
)
for column in [
    "full_document_count", "predicted_hostile_document_count",
    "predicted_hate_document_count",
]:
    term_validation_df[column] = pd.to_numeric(
        term_validation_df[column], errors="coerce"
    ).fillna(0).astype(int)

candidate_terms_df = term_validation_df[
    term_validation_df["document_count"].ge(MIN_MANUAL_TERM_DOCUMENTS)
    & term_validation_df["hostility_precision"].ge(MIN_HOSTILITY_PRECISION)
    & term_validation_df["hostility_lift"].ge(MIN_HOSTILITY_LIFT)
].copy()
candidate_terms_df["candidate_tier"] = np.select(
    [
        candidate_terms_df["hostility_precision"].ge(MIN_HIGH_CONFIDENCE_PRECISION),
        candidate_terms_df["hostility_precision"].ge(0.75),
    ],
    ["high_manual_association", "supported_manual_association"],
    default="exploratory_association",
)
candidate_terms_df["manual_confirm_offensive"] = ""
candidate_terms_df["manual_review_notes"] = ""
candidate_terms_df = candidate_terms_df.sort_values(
    ["candidate_tier", "full_document_count", "document_count"],
    ascending=[True, False, False],
)

high_association_terms_df = candidate_terms_df[
    candidate_terms_df["candidate_tier"].eq("high_manual_association")
].copy()

term_index_df.to_csv(REPORTS_DIR / "categorized_lexicon_term_index.csv", index=False)
term_validation_df.to_csv(REPORTS_DIR / "offensive_term_manual_validation.csv", index=False)
candidate_terms_df.to_csv(REPORTS_DIR / "candidate_offensive_terms_for_review.csv", index=False)
high_association_terms_df.to_csv(REPORTS_DIR / "high_manual_association_terms.csv", index=False)

print("Términos categorizados indexados:", len(term_index_df))
print("Candidatos con soporte manual:", len(candidate_terms_df))
print("Alta asociación manual:", len(high_association_terms_df))
display(candidate_terms_df.head(30))


In [ ]:
def save_horizontal_bar(data, label_column, value_column, title, filename, color):
    plot_data = data.head(TOP_N).sort_values(value_column)
    if plot_data.empty:
        print(f"[SKIP] Sin datos para {filename}")
        return None
    height = max(5, 0.30 * len(plot_data) + 2)
    fig, ax = plt.subplots(figsize=(11, height))
    ax.barh(plot_data[label_column], plot_data[value_column], color=color)
    ax.set_title(title)
    ax.set_xlabel(value_column.replace("_", " "))
    ax.set_ylabel("")
    fig.tight_layout()
    output_path = FIGURES_DIR / filename
    fig.savefig(output_path, dpi=180)
    plt.show()
    print("[OK]", output_path)
    return output_path


ranked_candidate_terms_df = candidate_terms_df.sort_values(
    ["full_document_count", "hostility_precision"], ascending=False
)
save_horizontal_bar(
    ranked_candidate_terms_df,
    "term_norm",
    "full_document_count",
    "Candidatos del lexicón: frecuencia en el corpus (requieren revisión)",
    "top_candidate_offensive_terms_full_corpus.png",
    "#9A4F3D",
)

ranked_high_association_df = high_association_terms_df.sort_values(
    ["full_document_count", "document_count"], ascending=False
)
save_horizontal_bar(
    ranked_high_association_df,
    "term_norm",
    "full_document_count",
    "Términos con respaldo manual fuerte: frecuencia en el corpus",
    "high_manual_association_terms_full_corpus.png",
    "#A6402D",
)

if not term_validation_df.empty:
    scatter_df = term_validation_df[term_validation_df["document_count"].ge(2)].copy()
    fig, ax = plt.subplots(figsize=(11, 7))
    sizes = 25 + 5 * np.sqrt(scatter_df["full_document_count"].clip(lower=0))
    points = ax.scatter(
        scatter_df["document_count"],
        scatter_df["hostility_precision"],
        s=sizes,
        c=scatter_df["hate_precision"],
        cmap="YlOrRd",
        alpha=0.72,
        edgecolor="#333333",
        linewidth=0.4,
    )
    ax.axhline(MIN_HOSTILITY_PRECISION, color="#555555", linestyle="--", linewidth=1)
    ax.set_xlabel("Documentos en muestra manual")
    ax.set_ylabel("Proporción manualmente hostil")
    ax.set_title("Validación manual de términos categorizados")
    colorbar = fig.colorbar(points, ax=ax)
    colorbar.set_label("Proporción con odio manual")
    fig.tight_layout()
    scatter_path = FIGURES_DIR / "term_manual_validation_scatter.png"
    fig.savefig(scatter_path, dpi=180)
    plt.show()
    print("[OK]", scatter_path)


## 3. Unigramas, bigramas y trigramas contrastados manualmente

El ranking usa frecuencia documental y log-odds suavizado. Un valor alto indica mayor asociación con comentarios manualmente hostiles, no que la expresión sea ofensiva en todos los contextos.


In [ ]:
ngram_contrast_tables = {}
for n_value, min_documents in [(1, 3), (2, 2), (3, 2)]:
    table = text_analysis.contrast_ngrams_by_binary_label(
        manual_df,
        text_column="text",
        document_column="tweet_id",
        label_column="manual_hostility_normalized",
        n=n_value,
        min_total_documents=min_documents,
    )
    ngram_contrast_tables[n_value] = table
    label = {1: "unigrams", 2: "bigrams", 3: "trigrams"}[n_value]
    table.to_csv(
        REPORTS_DIR / f"manual_hostility_{label}_contrast.csv", index=False
    )

manual_hostile_df = manual_df[
    manual_df["manual_hostility_normalized"].eq(1)
].copy()
manual_hostile_bigrams_df = text_analysis.ngram_document_frequency(
    manual_hostile_df, "text", "tweet_id", n=2, min_count=2
)
manual_hostile_trigrams_df = text_analysis.ngram_document_frequency(
    manual_hostile_df, "text", "tweet_id", n=3, min_count=2
)
manual_hostile_bigrams_df.to_csv(
    REPORTS_DIR / "manual_hostile_bigram_frequency.csv", index=False
)
manual_hostile_trigrams_df.to_csv(
    REPORTS_DIR / "manual_hostile_trigram_frequency.csv", index=False
)

for n_value, table in ngram_contrast_tables.items():
    plot_table = table[
        table["hostile_document_count"].ge(2)
        & table["log_odds_hostility"].gt(0)
    ].sort_values(["log_odds_hostility", "hostile_document_count"], ascending=False)
    ngram_name = {1: "Unigramas", 2: "Bigramas", 3: "Trigramas"}[n_value]
    save_horizontal_bar(
        plot_table,
        "ngram",
        "log_odds_hostility",
        f"{ngram_name} más asociados a hostilidad manual",
        f"manual_hostility_{n_value}gram_log_odds.png",
        "#496C78",
    )


## 4. N-gramas del corpus predicho como hostil

Para reducir ruido, se conservan únicamente bigramas y trigramas que contienen al menos un token de la lista de alta asociación manual. Esta capa sigue dependiendo de predicciones del modelo y es exploratoria.


In [ ]:
validated_tokens = set()
for term in high_association_terms_df["term_norm"].dropna().astype(str):
    validated_tokens.update(text_analysis.normalize_lexical_text(term).split())

predicted_hostile_df = corpus_df[corpus_df["ml_hostility_pred"].eq(1)].copy()
predicted_hostile_bigrams_df = text_analysis.ngram_document_frequency(
    predicted_hostile_df,
    text_column="text",
    document_column="tweet_id",
    n=2,
    required_tokens=validated_tokens,
    min_count=3,
)
predicted_hostile_trigrams_df = text_analysis.ngram_document_frequency(
    predicted_hostile_df,
    text_column="text",
    document_column="tweet_id",
    n=3,
    required_tokens=validated_tokens,
    min_count=3,
)
predicted_hostile_bigrams_df.to_csv(
    REPORTS_DIR / "predicted_hostile_offensive_bigrams.csv", index=False
)
predicted_hostile_trigrams_df.to_csv(
    REPORTS_DIR / "predicted_hostile_offensive_trigrams.csv", index=False
)

save_horizontal_bar(
    predicted_hostile_bigrams_df,
    "ngram",
    "document_count",
    "Bigramas ofensivos candidatos en comentarios predichos como hostiles",
    "predicted_hostile_offensive_bigrams.png",
    "#A36A36",
)
save_horizontal_bar(
    predicted_hostile_trigrams_df,
    "ngram",
    "document_count",
    "Trigramas ofensivos candidatos en comentarios predichos como hostiles",
    "predicted_hostile_offensive_trigrams.png",
    "#7C5D86",
)

print("Tokens de alta asociación usados como ancla:", len(validated_tokens))
print("Bigramas exportados:", len(predicted_hostile_bigrams_df))
print("Trigramas exportados:", len(predicted_hostile_trigrams_df))


## 5. Distribución por evento

El mapa de calor muestra frecuencia documental, no número total de repeticiones. Se limita a términos con alta asociación manual para evitar que palabras comunes del lexicón dominen la figura.


In [ ]:
high_term_set = set(high_association_terms_df["term_norm"])
event_term_df = (
    full_matches_df[full_matches_df["term_norm"].isin(high_term_set)]
    .groupby(["event_id", "term_norm"], as_index=False)
    .agg(document_count=("tweet_id", "nunique"))
)
event_term_df.to_csv(REPORTS_DIR / "high_association_terms_by_event.csv", index=False)

if not event_term_df.empty:
    top_event_terms = (
        event_term_df.groupby("term_norm")["document_count"]
        .sum()
        .nlargest(15)
        .index
    )
    event_pivot = (
        event_term_df[event_term_df["term_norm"].isin(top_event_terms)]
        .pivot(index="term_norm", columns="event_id", values="document_count")
        .fillna(0)
    )
    event_pivot.to_csv(REPORTS_DIR / "high_association_terms_by_event_matrix.csv")

    fig, ax = plt.subplots(figsize=(13, max(6, 0.45 * len(event_pivot))))
    image = ax.imshow(event_pivot.values, aspect="auto", cmap="YlOrBr")
    ax.set_xticks(range(len(event_pivot.columns)), labels=event_pivot.columns, rotation=40, ha="right")
    ax.set_yticks(range(len(event_pivot.index)), labels=event_pivot.index)
    ax.set_title("Términos de alta asociación manual por evento")
    ax.set_xlabel("")
    ax.set_ylabel("")
    colorbar = fig.colorbar(image, ax=ax)
    colorbar.set_label("Comentarios con el término")
    fig.tight_layout()
    heatmap_path = FIGURES_DIR / "high_association_terms_by_event_heatmap.png"
    fig.savefig(heatmap_path, dpi=180)
    plt.show()
    print("[OK]", heatmap_path)


## 6. Resumen y advertencias

- Las frecuencias no equivalen a severidad.
- La muestra manual está enriquecida por lexicón, por lo que las precisiones no estiman prevalencia poblacional.
- Las coincidencias pueden ser citas, negaciones, usos irónicos o palabras polisémicas.
- Los n-gramas del corpus completo dependen de predicciones de hostilidad.
- `manual_confirm_offensive` queda vacío para permitir revisión sustantiva de los candidatos antes de citarlos como vocabulario ofensivo consolidado.


In [ ]:
summary_df = pd.DataFrame([
    {"metric": "corpus_rows", "value": len(corpus_df)},
    {"metric": "manual_rows", "value": len(manual_df)},
    {"metric": "manual_hostility_rate", "value": round(base_hostility_rate, 4)},
    {"metric": "categorized_lexicon_terms", "value": len(term_index_df)},
    {"metric": "full_categorized_term_document_pairs", "value": len(full_matches_df)},
    {"metric": "candidate_terms_for_review", "value": len(candidate_terms_df)},
    {"metric": "high_manual_association_terms", "value": len(high_association_terms_df)},
    {"metric": "predicted_hostile_rows", "value": len(predicted_hostile_df)},
    {"metric": "predicted_hostile_offensive_bigrams", "value": len(predicted_hostile_bigrams_df)},
    {"metric": "predicted_hostile_offensive_trigrams", "value": len(predicted_hostile_trigrams_df)},
    {"metric": "methodological_status", "value": "candidate_terms_require_contextual_review"},
])
summary_path = REPORTS_DIR / "lexical_analysis_summary.csv"
summary_df.to_csv(summary_path, index=False)
print("[OK]", summary_path)
print("Tablas:", REPORTS_DIR)
print("Figuras:", FIGURES_DIR)
display(summary_df)
